In [1]:
import pandas as pd
from PIL import Image

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import torch.nn as nn

from sklearn.metrics import f1_score
import numpy as np

from training_monitor import TrainingMonitor

<jemalloc>: Unsupported system page size


In [2]:
import torch, gc, random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=2)

In [3]:
train1_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_path   = "/datasets/multi-view-pig-posture-recognition/test_images"

train1_csv  = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv  = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv    = "/datasets/multi-view-pig-posture-recognition/test.csv"

import pandas as pd

train1_df = pd.read_csv(train1_csv)
train2_df = pd.read_csv(train2_csv)
test_df   = pd.read_csv(test_csv)

train1_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


In [4]:
import ast
from PIL import Image
from torch.utils.data import Dataset

class PigDataset(Dataset):
    def __init__(self, df, img_root, transform=None, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = f"{self.img_root}/{row['image_id']}"
        img = Image.open(img_path).convert("RGB")

        # bbox ist String → "[967.5,331.5,463.0,447.0]"
        bbox = row["bbox"]
        if isinstance(bbox, str):
            bbox = ast.literal_eval(bbox)

        xmin, ymin, w, h = bbox

        # sauber runden
        x1 = int(round(xmin))
        y1 = int(round(ymin))
        x2 = int(round(xmin + w))
        y2 = int(round(ymin + h))

        # optional clippen
        W, H = img.size
        x1 = max(0, min(x1, W - 1))
        x2 = max(1, min(x2, W))
        y1 = max(0, min(y1, H - 1))
        y2 = max(1, min(y2, H))

        img = img.crop((x1, y1, x2, y2))

        if self.transform:
            img = self.transform(img)

        if self.has_labels:
            label = int(row["class_id"])
            return img, label
        else:
            return img, row["row_id"]  # für Submission später

In [5]:
import torchvision.transforms as T

train_tfms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

val_tfms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

In [6]:
from sklearn.model_selection import train_test_split

train1_train_df, train1_val_df = train_test_split(
    train1_df,
    test_size=0.2,
    random_state=42,
    stratify=train1_df["class_id"]  # wichtig für Macro F1!
)

train2_train_df, train2_val_df = train_test_split(
    train2_df,
    test_size=0.2,
    random_state=42,
    stratify=train2_df["class_id"]
)

print("T1 sizes:", len(train1_train_df), len(train1_val_df))
print("T2 sizes:", len(train2_train_df), len(train2_val_df))

T1 sizes: 18347 4587
T2 sizes: 18760 4690


In [7]:
train1_train_ds = PigDataset(train1_train_df, train1_path, transform=train_tfms, has_labels=True)
train1_val_ds   = PigDataset(train1_val_df,   train1_path, transform=val_tfms,   has_labels=True)

train1_train_loader = DataLoader(train1_train_ds, batch_size=8, shuffle=True,  num_workers=0, pin_memory=False)
train1_val_loader   = DataLoader(train1_val_ds,   batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()

def freeze_backbone(model):
    for name, p in model.named_parameters():
        if "fc" not in name:
            p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

freeze_backbone(model)

/opt/conda/envs/torch/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    n = 0
    preds_all = []
    targs_all = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = loss_fn(logits, yb)

        bs = xb.size(0)
        total_loss += float(loss.item()) * bs
        n += bs

        preds = logits.argmax(dim=1)
        preds_all.append(preds.detach().cpu().numpy())
        targs_all.append(yb.detach().cpu().numpy())

    preds_all = np.concatenate(preds_all)
    targs_all = np.concatenate(targs_all)

    return {
        "val_loss": float(total_loss / max(1, n)),
        "val_f1": float(f1_score(targs_all, preds_all, average="macro")),
    }

In [10]:
from training_monitor import TrainingMonitor

def train_baseline(
    model,
    train_loader,
    val_loader,
    loss_fn,
    device,
    epochs=3,
    lr=1e-4,
    weight_decay=1e-3,
    use_onecycle=False,
):
    # Optimizer nur auf trainierbare Parameter
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)

    if use_onecycle:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=total_steps,
            pct_start=0.3,
            anneal_strategy="cos",
            div_factor=25.0,
            final_div_factor=1e4,
        )
    else:
        scheduler = None

    groups = {
        "Loss": ["train_loss", "val_loss"],
        "F1": ["val_f1"],
        "Learning Rate": ["lr"],
    }

    monitor = TrainingMonitor(
        total_iterations=epochs,   # 1 update pro Epoche
        plot_mode="separate",
        metric_groups=groups,
        zoom=True,
        row_height=1.5
    )

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        n = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            bs = xb.size(0)
            running_loss += float(loss.item()) * bs
            n += bs

        train_loss = float(running_loss / max(1, n))
        val_metrics = evaluate(model, val_loader, loss_fn, device)

        current_lr = scheduler.get_last_lr()[0] if scheduler is not None else optimizer.param_groups[0]["lr"]

        monitor.update({
            "train_loss": train_loss,
            "val_loss": float(val_metrics["val_loss"]),
            "val_f1": float(val_metrics["val_f1"]),
            "lr": float(current_lr),
        })

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} | "
            f"val_f1={val_metrics['val_f1']:.4f} | "
            f"lr={current_lr:.2e}"
        )

        if val_metrics["val_f1"] > best_val_f1:
            best_val_f1 = val_metrics["val_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_f1

In [ ]:
best_state, best_f1 = train_baseline(
    model=model,
    train_loader=train1_train_loader,
    val_loader=train1_val_loader,      
    loss_fn=loss_fn,
    device=device,
    epochs=3,
    lr=1e-4,
    use_onecycle=False
)

print("Best val_f1:", best_f1)